# Weather-or-Not: Flight Delay Classifier
### Naive Bayes vs. Feedforward Neural Network

**Target:** Predict whether a flight will arrive delayed (≥15 min) based on weather at origin & destination airports.

**Pipeline:**
1. Load flight + weather data
2. Merge weather onto flights (origin & destination)
3. Feature engineering
4. Train / test split
5. Model with FFN


## 0. Imports & Config

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from datetime import timedelta

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, precision_score, recall_score,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
)
from sklearn.calibration import CalibrationDisplay
from sklearn.naive_bayes import GaussianNB, CategoricalNB, ComplementNB
from sklearn.neural_network import MLPClassifier

import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)

with open('params_file.json') as f:
    params = json.load(f)

FLIGHT_CSV      = params['Output_Directory'] + 'flights_dataset.csv'
IEM_WEATHER_DIR = params['Dataset_Directory'] + '2024_iem_weather'
DELAY_THRESHOLD = 15        # FAA standard: 15+ min = delayed
RANDOM_STATE    = 1354
TEST_SIZE       = 0.2

Config loaded ✓


## 1. Load Data

In [ ]:
%%time
#Same loading as with NB
CACHE_PATH = 'flights_with_weather.parquet'

if os.path.exists(CACHE_PATH):
    print('Loading cached merged dataset...')
    merged = pd.read_parquet(CACHE_PATH)
else:
    print('Joining weather onto flights (this may take a few minutes)...')
    weather_records = []
    for _, row in flights.iterrows():
        orig_wx = get_interpolated_weather(row.get('ORIGIN',''), row['DEP_TS'], iem_weather, 'ORIG')
        dest_wx = get_interpolated_weather(row.get('DEST',''),   row['ARR_TS'], iem_weather, 'DEST')
        weather_records.append({**orig_wx, **dest_wx})
    wx_df  = pd.DataFrame(weather_records, index=flights.index)
    merged = pd.concat([flights, wx_df], axis=1)
    merged.to_parquet(CACHE_PATH, index=False)
    print(f'Cached to {CACHE_PATH}')

print(f'Merged shape: {merged.shape}')

Loading cached merged dataset...
Merged shape: (6510337, 58)
CPU times: total: 8.53 s
Wall time: 2.4 s


## 4. Feature Engineering

In [ ]:
# Temporal features
WEATHER_COLS = ['tmpf', 'dwpf', 'relh', 'drct', 'sknt', 'vsby', 'mslp', 'gust']

merged['DEP_HOUR']   = merged['DEP_TS'].dt.hour
merged['DEP_DOW']    = merged['DEP_TS'].dt.dayofweek
merged['DEP_MONTH']  = merged['DEP_TS'].dt.month
merged['IS_WEEKEND'] = (merged['DEP_DOW'] >= 5).astype(int)
merged['RUSH_HOUR']  = merged['DEP_HOUR'].apply(
    lambda h: 1 if h in range(7, 10) or h in range(16, 20) else 0
)

#Grab next holdays distance
def get_days_to_next_holiday(df, timestamp_col='DEP_TS'):
    print(df[timestamp_col][0])
    print(pd.to_datetime(df[timestamp_col][0]))
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    #only 2024 for our dataset but good ot be sure
    years = df[timestamp_col].dt.year.unique()
    #only US holidays as this is a US databse
    print(years)
    us_holidays = holidays.US(years=list(years) + [max(years) + 1])
    holiday_dates = sorted(us_holidays.keys())
    holiday_series = pd.to_datetime(holiday_dates)

    def find_next(ts):
        future_holidays = holiday_series[holiday_series >= ts.normalize()]
        if not future_holidays.empty:
            #grab closest holiday in the future
            next_h = future_holidays[0]
            return (next_h - ts).days
        return np.nan

    return merged[timestamp_col].apply(find_next)

#Flight Denisty Funciton (will be slightly difficult to get from flight aware API)
def add_flight_density(df):
    #Get the orign/
    df = df.sort_values(['ORIGIN', 'DEP_TS'])
    
    #Grab flights within a 2 hour window at that airport
    density = (
        df.set_index('DEP_TS')
        .groupby('ORIGIN')['FLIGHTS'] # 'FLIGHTS' column is usually just 1s
        .rolling('2h', center=True)
        .count()
        .reset_index(drop=True)
    )
    
    df['FLIGHT_DENSITY'] = density.values
    return df

# Feature lists
CONTINUOUS_FEATURES = [
    'ORIG_tmpf','ORIG_dwpf','ORIG_relh','ORIG_sknt','ORIG_vsby','ORIG_mslp',
    'DEST_tmpf','DEST_dwpf','DEST_relh','DEST_sknt','DEST_vsby','DEST_mslp',
    'DELTA_tmpf','DELTA_sknt','DELTA_vsby',
    'DEP_HOUR', 'DAYS_TO_HOLIDAY', 'FLIGHT_DENSITY'
]

#Theese will need embeddings
CATEGORICAL_FEATURES = [
    'DEP_DOW','DEP_MONTH','IS_WEEKEND','RUSH_HOUR',
    'ORIG_LOW_VIS','ORIG_HIGH_WIND','ORIG_GUSTING',
    'DEST_LOW_VIS','DEST_HIGH_WIND','DEST_GUSTING',
]
ALL_FEATURES = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES
TARGET = 'DELAYED'

CACHE_PATH = 'flights_with_weather_features.parquet'

if os.path.exists(CACHE_PATH):
    print('Loading cached merged dataset...')
    merged = pd.read_parquet(CACHE_PATH)
else:
    # Apply to dataframe
    print("Days To Holiday Feature:")
    %time
    merged['DAYS_TO_HOLIDAY'] = get_days_to_next_holiday(merged, 'DEP_TS')
    print("Flight Density Feature:")
    %time
    merged = add_flight_density(merged)

    print("Weather Delta Features")
    # Weather delta (destination - origin)
    for col in WEATHER_COLS:
        o, d = f'ORIG_{col}', f'DEST_{col}'
        if o in merged.columns and d in merged.columns:
            merged[f'DELTA_{col}'] = merged[d] - merged[o]

    print("Weather Flag Features")
    # Derived flags
    for pfx in ['ORIG', 'DEST']:
        merged[f'{pfx}_LOW_VIS']   = (merged[f'{pfx}_vsby'] < 3).astype(float)
        merged[f'{pfx}_HIGH_WIND'] = (merged[f'{pfx}_sknt'] > 20).astype(float)
        merged[f'{pfx}_GUSTING']   = merged[f'{pfx}_gust'].notna().astype(float)

merged.to_parquet(CACHE_PATH, index=False)
print(f'Cached to {CACHE_PATH}')


model_df = merged.copy()
print(f'Modelling dataset: {len(model_df):,} rows | Delay rate: {model_df[TARGET].mean():.1%}')

if 'CANCELLED' in model_df.columns:
    model_df = model_df[model_df['CANCELLED'] == 0]
if 'DIVERTED' in model_df.columns:
    model_df = model_df[model_df['DIVERTED'] == 0]
model_df = model_df[ALL_FEATURES + [TARGET]].dropna(subset=[TARGET])

Loading cached merged dataset...
Cached to flights_with_weather_features.parquet
Modelling dataset: 6,406,581 rows | Delay rate: 21.0%


MemoryError: Unable to allocate 2.33 GiB for an array with shape (48, 6510337) and data type float64

In [ ]:
%%time
CACHE_PATH = 'flights_with_weather_features.parquet'

if os.path.exists(CACHE_PATH):
    print('Loading cached merged dataset...')a
    merged = pd.read_parquet(CACHE_PATH)
else:
    print('Joining weather onto flights (this may take a few minutes)...')
    weather_records = []
    for _, row in flights.iterrows():
        orig_wx = get_interpolated_weather(row.get('ORIGIN',''), row['DEP_TS'], iem_weather, 'ORIG')
        dest_wx = get_interpolated_weather(row.get('DEST',''),   row['ARR_TS'], iem_weather, 'DEST')
        weather_records.append({**orig_wx, **dest_wx})
    wx_df  = pd.DataFrame(weather_records, index=flights.index)
    merged = pd.concat([flights, wx_df], axis=1)
    merged.to_parquet(CACHE_PATH, index=False)
    print(f'Cached to {CACHE_PATH}')

print(f'Merged shape: {merged.shape}')

In [9]:
merged.columns

Index(['Unnamed: 0', 'YEAR', 'FL_DATE', 'OP_UNIQUE_CARRIER',
       'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER', 'TAIL_NUM', 'OP_CARRIER_FL_NUM',
       'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID',
       'ORIGIN', 'ORIGIN_WAC', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID',
       'DEST_CITY_MARKET_ID', 'DEST', 'DEST_WAC', 'DEP_TIME', 'DEP_DELAY',
       'DEP_DELAY_GROUP', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_GROUP',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'AIR_TIME', 'FLIGHTS',
       'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
       'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'source_file',
       'ORIGIN_WEATHER_STATION', 'DEST_WEATHER_STATION', 'DELAYED', 'DEP_TS',
       'ARR_TS', 'FL_DATE_CLEAN', 'ORIG_tmpf', 'ORIG_dwpf', 'ORIG_relh',
       'ORIG_drct', 'ORIG_sknt', 'ORIG_vsby', 'ORIG_mslp', 'ORIG_gust',
       'DEST_tmpf', 'DEST_dwpf', 'DEST_relh', 'DEST_drct', 'DEST_sknt',
       'DEST_vsby', 'DEST_mslp', 'DEST_gust', 'DEP_HOUR', 'DEP_D

## 5. Output Labels

In [24]:
merged['DEST_gust'].unique()

array([nan])

## 6. Train / Test Split

In [14]:
X = model_df[ALL_FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train delay rate: {y_train.mean():.1%}  |  Test delay rate: {y_test.mean():.1%}')

Train: 5,125,264  |  Test: 1,281,317
Train delay rate: 21.0%  |  Test delay rate: 21.0%
